In [4]:
import os
import pandas as pd
import re
from bisect import bisect_right
from transformers import AutoTokenizer

# ----------------------------
# SETTINGS
# ----------------------------
WIKI_TREATIES = "./wiki-treaties_formatted.csv"
UNO_TREATIES = "./UNO-Treaties.csv"
OHCHR_INSTRUMENTS = "../ohchr_instruments/ohchr_instruments_detailed-instit.csv"
CONV_PROT_REC = "../conv-prot-rec/conventions-protocols-recommendations.csv"
RESOLUTIONS = "../resolutions/ga_resolutions_1946_2019.csv"

df_wiki = pd.read_csv(WIKI_TREATIES)
df_uno = pd.read_csv(UNO_TREATIES)
df_ohchr_ins = pd.read_csv(OHCHR_INSTRUMENTS)
df_conv = pd.read_csv(CONV_PROT_REC)
df_res_all = pd.read_csv(RESOLUTIONS)

# ----------------------------
# TOKENIZING
# ----------------------------
MAX_TOKENS = 2100
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B")

In [8]:
def trim_contents(text: str, tokenizer, target_tokens: int, max_tokens: int) -> str | None:
    """Shorten `text` to end on a sentence boundary at/after `target_tokens`.
    Returns the original text unchanged if already short enough, a trimmed
    candidate if it fits within `max_tokens`, or None if it can't be brought
    under budget at all (caller should drop the document)."""
    encoded = tokenizer.encode(text, add_special_tokens=False)
    total_tokens = len(encoded)

    if total_tokens <= target_tokens:
        return text

    # NOTE: removed the old early `if total_tokens > max_tokens: return None`.
    # That check fired *before* trimming was attempted, so any document
    # longer than max_tokens was dropped instead of being trimmed. The
    # max_tokens check belongs only on the trimmed candidate, below.

    sentence_endings = [m.end() for m in re.finditer(r"[.!?]", text)]

    first_part = tokenizer.decode(encoded[:target_tokens])
    char_pos = len(first_part)

    next_end = next((pos for pos in sentence_endings if pos >= char_pos), None)
    if next_end is None:
        return None

    candidate = text[:next_end]
    if len(tokenizer.encode(candidate, add_special_tokens=False)) <= max_tokens:
        return candidate
    return None


def trim(
    df: pd.DataFrame,
    tokenizer,
    target_tokens: int = MAX_TOKENS-20,
    max_tokens: int = MAX_TOKENS,
    content_col: str = "content",
    id_col: str = "res_id2",
) -> pd.DataFrame:
    """Trim `content_col` in-place (on a copy) to ~target_tokens, ending on a
    sentence boundary. Rows that can't be brought under max_tokens are dropped.
    Returns a new, filtered DataFrame — no tokenized data is stored anywhere,
    only plain trimmed strings."""
    df = df.copy()
    print(f"Initial dataframe length: {len(df)}")

    df[content_col] = df[content_col].apply(
        lambda t: trim_contents(t, tokenizer, target_tokens, max_tokens)
    )

    dropped_mask = df[content_col].isna()
    if dropped_mask.any():
        dropped_ids = df.loc[dropped_mask, id_col].tolist()
        print(f"Dropping {len(dropped_ids)} rows that couldn't be trimmed under {max_tokens} tokens.")

    return df.loc[~dropped_mask].reset_index(drop=True)


df_res = trim(
    df_res_all,
    tokenizer,  # <- define this before running
    target_tokens=2100,
    max_tokens=2200,  # small buffer so sentence-boundary rounding has room
    content_col="content",
    id_col="res_id2",)

df_res.head(5)

Initial dataframe length: 17913
Dropping 656 rows that couldn't be trimmed under 2200 tokens.


,res_id2,res_id3,res_id2_unlet,alt_id_dic,session_type,session_reg,session_sp,session_es,resn,res_letter,date_p,date_c,filename,content,location,record,draft,topic,n_inc_cit
0,996 (es-i and ii),996 (es-i and ii) of 9 november 1956,996 (es-i and ii),0,emergency,NaN,NaN,1.0,996,NaN,1956-11-09,9 november 1956,ga_es_1_996.txt,996 (es-i and ii). credentials of representati...,cred. cttee,a/pv.571,a/3321,credentials of representatives to the 1st and ...,0
1,997 (es-i),997 (es-i) of 2 november 1956,997 (es-i),0,emergency,NaN,NaN,1.0,997,NaN,1956-11-02,2 november 1956,ga_es_1_997.txt,"997 (es-i). the general assembly, noting the d...",plenary,a/pv.562,a/3256,question considered by the security council at...,9
2,998 (es-i),998 (es-i) of 4 november 1956,998 (es-i),0,emergency,NaN,NaN,1.0,998,NaN,1956-11-04,4 november 1956,ga_es_1_998.txt,"998 (es-i). the general assembly, bearing in m...",plenary,a/pv.563,a/3276,question considered by the security council at...,5
3,999 (es-i),999 (es-i) of 4 november 1956,999 (es-i),0,emergency,NaN,NaN,1.0,999,NaN,1956-11-04,4 november 1956,ga_es_1_999.txt,"999 (es-i). the general assembly, noting with ...",plenary,a/pv.563,a/3275,question considered by the security council at...,4
4,1000 (es-i),1000 (es-i) of 5 november 1956,1000 (es-i),0,emergency,NaN,NaN,1.0,1000,NaN,1956-11-05,5 november 1956,ga_es_1_1000.txt,"1000 (es-i). the general assembly, having requ...",plenary,a/pv.565,a/3290,question considered by the security council at...,6


In [ ]:
print(f"WIKI_TREATIES columns: {df_wiki.columns}")
print(f"UNO_TREATIES columns: {df_uno.columns}")
print(f"OHCHR_INSTRUMENTS columns: {df_ohchr_ins.columns}")
print(f"CONV_PROT_REC columns: {df_conv.columns}")
print(f"RESOLUTIONS columns: {df_res.columns}")

In [42]:
print(f"{'WIKI_TREATIES':=^30}")
print(f"Number of unique cleaned titles: {df_wiki['cleaned_title'].nunique()}\n"
      f"Number of all titles: {len(df_wiki['cleaned_title'])}")
print(f"\nNumber of unique cleaned alternative names: {df_wiki['cleaned_note'].nunique()}\n"
      f"Length of all alternative names: {len(df_wiki['cleaned_note'].dropna())}")


print("Non-unique cleaned titles:")
non_unique_titles = df_wiki[df_wiki.duplicated(subset='cleaned_title', keep=False)]
non_unique_titles[['year', 'cleaned_title', 'name']]

========WIKI_TREATIES=========
Number of unique cleaned titles: 345
Number of all titles: 364

Number of unique cleaned alternative names: 101
Length of all alternative names: 102
Non-unique cleaned titles:


,year,cleaned_title,name
21,1905,Japan–Korea Treaty,Japan–Korea Treaty of 1905
28,1910,Japan–Korea Treaty,Japan–Korea Treaty of 1910
34,1913,Treaty of London,Treaty of London (1913)
35,1913,Treaty of Bucharest,Treaty of Bucharest (1913)
40,1915,Treaty of London,Treaty of London (1915) (London Pact)
44,1916,Treaty of Bucharest,Treaty of Bucharest (1916)
51,1918,Treaty of Bucharest,Treaty of Bucharest (1918)
64,1920,Treaty of Warsaw,Treaty of Warsaw (1920)
66,1920,Treaty of Rapallo,Treaty of Rapallo (1920)
67,1920,Treaty of Moscow,Treaty of Moscow (1920)


In [43]:
print(f"{'UNO_TREATIES':=^50}")
print(f"Number of unique titles: {df_uno['title'].nunique()}\n\
        Length of all titles: {len(df_uno['title'])}")

dupes = (
    df_uno.groupby(["title", "location"])
          .size()
          .reset_index(name="count")
          .query("count > 1")
)
print('Duplicates considering the tile and the loc:')
print(dupes, "\n\n\n")

print(f"{'INSTRUMENTS':=^50}")
print(f"Number of unique titles: {df_ins['title'].nunique()}\n\
        Length of all titles: {len(df_ins['title'])}")

dupes = (
    df_ins.groupby(["title", "adoption_date"])
          .size()
          .reset_index(name="count")
          .query("count > 1")
)

print('Duplicates considering the tile and the date:')
print(dupes, "\n\n\n")

print(f"{'ILO CONVENTIONS AND PROTOCOLS':=^50}")
print(f"Number of unique titles: {df_conv['title'].nunique()}\n\
        Length of all titles: {len(df_conv['title'])}")

dupes = (
    df_conv.groupby(["title", "year"])
          .size()
          .reset_index(name="count")
          .query("count > 1")
)

print('Duplicates considering the tile and the date:')
print(dupes)


===================UNO_TREATIES===================
Number of unique titles: 398
        Length of all titles: 428
Duplicates considering the tile and the loc:
                                                 title  location  count
74   Amendments to the Convention on the Internatio...    London      2
82   Amendments to the title and substantive provis...    London      2
185  Customs Convention on the International Transp...    Geneva      2
194  European Agreement concerning the Work of Crew...    Geneva      2
231                      International Cocoa Agreement    Geneva      5
234                     International Coffee Agreement    London      2
235                     International Coffee Agreement  New York      3
269             International Natural Rubber Agreement    Geneva      2
274                      International Sugar Agreement    Geneva      4
276                      International Sugar Agreement  New York      2 



===================INSTRUMENTS===============

In [44]:
'''
De los titulos que si son unicos, podemos usarlos directamente, sin especificar el año.
De los titulos que no son unicos, tenemos que usar el año. Podemos usarlo de las siguientes maneras
- El titulo seguido de "of YEAR"
- El titulo seguido de (YEAR)
- El titulo precedido por YEAR
- El titulo precedido por (YEAR)
Solo el titulo, extraer el fragmento con un padding de 10 caracteres y decidir
'''

'\nDe los titulos que si son unicos, podemos usarlos directamente, sin especificar el año.\nDe los titulos que no son unicos, tenemos que usar el año. Podemos usarlo de las siguientes maneras\n- El titulo seguido de "of YEAR"\n- El titulo seguido de (YEAR)\n- El titulo precedido por YEAR\n- El titulo precedido por (YEAR)\nSolo el titulo, extraer el fragmento con un padding de 10 caracteres y decidir\n'

In [45]:
# Lowercase, otherwise nothing is detected
df_wiki["cleaned_title"] = df_wiki["cleaned_title"].str.lower()

# -------------------------
# Separate unique/non-unique
# -------------------------
unique_titles = df_wiki.drop_duplicates("cleaned_title", keep=False)

non_unique_titles = df_wiki[
    df_wiki.duplicated("cleaned_title", keep=False)
]

# -------------------------
# Collect matches
# -------------------------
MAX_WORD_DIST = 6

records = []

# --- 1. Build map: pattern -> (treaty_id, treaty_title) ---
all_known = []  # (pattern, treaty_id, title)

# Unique titles (no year needed at detection)
for _, t in unique_titles.iterrows():
    all_known.append((
        re.escape(t["cleaned_title"]),
        f'{t["cleaned_title"]};{t["year"]}',
        t["cleaned_title"],
    ))

# Non-unique titles (allow year within MAX_WORD_DIST words)
for _, t in non_unique_titles.iterrows():
    year = str(int(t["year"]))
    title = t["cleaned_title"]

    pat = (
        rf"\b{re.escape(title)}\b(?:\W+\w+){{0,{MAX_WORD_DIST}}}\W+{year}\b"
        rf"|"
        rf"\b{year}\b(?:\W+\w+){{0,{MAX_WORD_DIST}}}\W+{re.escape(title)}\b"
        rf"|"
        rf"\b{re.escape(title)}\s*\({year}\)"
        rf"|"
        rf"\({year}\)\s*{re.escape(title)}"
        rf"|"
        rf"\b{re.escape(title)}\s+of\s+{year}\b"
    )

    all_known.append((pat, f"{title};{year}", title))

# repite para df_1, df_2, df_3 cuando los tengas (con su propio identifier/URL)

# --- 2. Compila UN solo regex con grupos nombrados (o usa un dict índice->id) ---
# Con muchos títulos, mejor usar índice numérico como nombre de grupo
combined_pattern = "|".join(f"(?P<t{i}>{p})" for i, (p, _, _) in enumerate(all_known))
combined_re = re.compile(combined_pattern)
id_lookup = {f"t{i}": (tid, title) for i, (_, tid, title) in enumerate(all_known)}


In [46]:
import os
import pandas as pd
import re
from bisect import bisect_right

# ----------------------------
# SETTINGS
# ----------------------------

df_wiki = pd.read_csv(WIKI_TREATIES)
df_uno = pd.read_csv(UNO_TREATIES)
df_ins = pd.read_csv(OHCHR_INSTRUMENTS)
df_conv = pd.read_csv(CONV_PROT_REC)
df_res = pd.read_csv(RESOLUTIONS)
df_uno["year"] = pd.to_datetime(df_uno["date"], format="%d %B %Y").dt.year

df_res = df_res.head(100)


MAX_WORD_DIST = 6


# ----------------------------------------------------
# Function to create regex patterns from a dataframe
# ----------------------------------------------------
def build_known_patterns(
    df,
    title_col,
    identifier_func,
    unique_by=None,
    allow_year_disambiguation=False,
    year_col="year"
):
    """
    Returns:
        list of tuples:
        (regex_pattern, identifier, title)
    """

    patterns = []

    # Default: everything is unique by title
    if unique_by is None:
        unique_rows = df
        non_unique_rows = pd.DataFrame(columns=df.columns)
    else:
        unique_rows = df.drop_duplicates(unique_by, keep=False)
        non_unique_rows = df[df.duplicated(unique_by, keep=False)]


    # -----------------------------
    # Unique titles
    # -----------------------------
    for _, row in unique_rows.iterrows():

        title = row[title_col].lower()

        patterns.append((
            rf"\b{re.escape(title)}\b",
            identifier_func(row),
            title
        ))


    # -----------------------------
    # Non unique titles
    # -----------------------------
    if allow_year_disambiguation:

        for _, row in non_unique_rows.iterrows():

            title = row[title_col].lower()
            year = str(int(row[year_col]))

            pat = (
                rf"\b{re.escape(title)}\b(?:\W+\w+){{0,{MAX_WORD_DIST}}}\W+{year}\b"
                rf"|"
                rf"\b{year}\b(?:\W+\w+){{0,{MAX_WORD_DIST}}}\W+{re.escape(title)}\b"
                rf"|"
                rf"\b{re.escape(title)}\s*\({year}\)"
                rf"|"
                rf"\({year}\)\s*{re.escape(title)}"
                rf"|"
                rf"\b{re.escape(title)}\s+of\s+{year}\b"
            )

            patterns.append((
                pat,
                identifier_func(row),
                title
            ))

    return patterns



# ----------------------------------------------------
# Build all known treaty/instrument patterns
# ----------------------------------------------------

all_known = []


# 1. Wikipedia treaties
all_known.extend(
    build_known_patterns(
        df_wiki,
        title_col="cleaned_title",
        unique_by="cleaned_title",
        allow_year_disambiguation=True,
        identifier_func=lambda r: f'{r["cleaned_title"]};{r["year"]}',
        year_col="year"
    )
)


# 2. UNO Treaties
# Titles are not always unique -> Title + Date
all_known.extend(
    build_known_patterns(
        df_uno,
        title_col="title",
        unique_by=["title", "date"],
        allow_year_disambiguation=True,
        identifier_func=lambda r: f'{r["title"]};{r["date"]}',
        year_col="year"
    )
)


# 3. OHCHR Instruments
# URL is unique
all_known.extend(
    build_known_patterns(
        df_ins,
        title_col="title",
        unique_by="title",
        allow_year_disambiguation=False,
        identifier_func=lambda r: r["url"]
    )
)


# 4. Conventions / Protocols / Recommendations
# Code is unique
all_known.extend(
    build_known_patterns(
        df_conv,
        title_col="title",
        unique_by="code",
        allow_year_disambiguation=False,
        identifier_func=lambda r: r["code"]
    )
)



# ----------------------------------------------------
# Compile ONE regex
# ----------------------------------------------------

combined_pattern = "|".join(
    f"(?P<t{i}>{pattern})"
    for i, (pattern, _, _) in enumerate(all_known)
)

combined_re = re.compile(
    combined_pattern,
    flags=re.IGNORECASE
)


id_lookup = {
    f"t{i}": (identifier, title)
    for i, (_, identifier, title) in enumerate(all_known)
}


In [47]:
records = []
candidates = []

KEYWORDS = [
    "agreement",
    "treaty",
    "convention",
    "protocol",
    "amendment",
    "charter",
    "covenant",
    "pact"
]

AVOID_KEYWORDS = [
    r"treaty series",
    r"treaty\s+series,\s*vol\.",
]

keyword_re = re.compile(
    r"\b(" + "|".join(KEYWORDS) + r")\b",
    re.IGNORECASE
)

avoid_re = re.compile(
    "|".join(AVOID_KEYWORDS),
    re.IGNORECASE
)


WINDOW_CHARS = 100

for _, res in df_res.iterrows():
    content = res["content"]
    res_id2 = res["res_id2"]
    res_part = res["part"]

    known_spans = []  # list of (start, end) already matched

    # --- Known treaty matches ---
    for m in combined_re.finditer(content):
        group_name = m.lastgroup
        tid, title = id_lookup[group_name]

        start = m.start()
        end = m.end()
        window = content[max(0, start - WINDOW_CHARS): min(len(content), end + WINDOW_CHARS)]

        records.append({
            "res_id2": res_id2,
            "res_part": res_part,
            "document": title,
            "id": tid,
            "start": start,
            "end": end,
            "context": window
        })

        known_spans.append((start, end))

    known_spans.sort()
    starts = [s for s, e in known_spans]

    # --- Standalone keyword matches not overlapping known treaties ---
    for km in keyword_re.finditer(content):
        k_start, k_end = km.start(), km.end()

        # Ignore "treaty series, vol." references
        nearby = content[max(0, k_start-30): k_end+30]

        if avoid_re.search(nearby):
            continue

        idx = bisect_right(starts, k_start) - 1
        overlapped = False

        for j in (idx, idx + 1):
            if 0 <= j < len(known_spans):
                s, e = known_spans[j]
                if s <= k_start < e or s < k_end <= e:
                    overlapped = True
                    break

        if not overlapped:
            window = content[max(0, k_start - WINDOW_CHARS): min(len(content), k_end + WINDOW_CHARS)]

            candidates.append({
                "res_id2": res_id2,
                "res_part": res_part,
                "keyword": km.group(),
                "start": k_start,
                "end": k_end,
                "context": window
            })

cites = pd.DataFrame(records).drop_duplicates().reset_index(drop=True)
unknown_candidates = pd.DataFrame(candidates).drop_duplicates().reset_index(drop=True)

KeyError: 'part'

In [ ]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
unknown_candidates

In [ ]:
cites

In [ ]:
pd.reset_option("display.max_colwidth")
pd.reset_option("display.max_rows")

"""
1. create a df with the unique titles (cleaned_name column)
2. scan them through the resolution['content]
3. save all in a cites dataframe
4. now the non unique titles (cleaned_name column)
5. scan them non alone through the resolution['content]: they must be accompagned with 'of df['year'], ' (df['year'])', 'df['year'] df['cleaned_name']', '(df['year']) df['cleaned_name']'
6. save everything in cites dataframe
Note: the cites dataframe is resolutions[res_id2], treaty and id, which is composed of the treaty + comma + year
"""


In [ ]:
cites.to_csv("treaty_citations-first-draft.csv", index=False)

In [ ]:
"""


Geneva convention. there are many, the first is just geneva convention but the other ones are named by second, third, forth geneva convention. The problem is that the cases where the forth is references, sometimes they dont inlcude the ordinal, so the system detects as the first, but the text next inlcude the date, including 1949
Solution, the geneva convention should also detects a date
Geneva Convention relative to the Protection of Civilian Persons in Time of War, of 12 August 1949 = fourth geneva convention


"""

In [ ]:
# group to see the most cited treaties
cites = (
    cites.groupby('id')
         .agg(count=('res_id2', 'count'))
         .reset_index()
)

In [ ]:
cites.sort_values(by='count', ascending=False, inplace=True)
print(cites.head(10))